# Breaking Defenses & Black-Box Attacks

In [ ]:
import torch
from torch import nn
from torch.optim import Adam
import torch.nn.functional as F
from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader

from torchvision import transforms
from torchvision.models import resnet18, mobilenet_v2
from torchvision.datasets.cifar import CIFAR10

from tqdm import trange, tqdm

torch.manual_seed(0)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

# CIFAR10 Dataset (5 points)

In [ ]:
norm_mean = (0.4914, 0.4822, 0.4465)
norm_std = (0.2023, 0.1994, 0.2010)
batch_size = 128

mu = torch.tensor(norm_mean).view(3,1,1).to(device)
std = torch.tensor(norm_std).view(3,1,1).to(device)

# TODO: Set the upper limit and lower limit possible for images
upper_limit = ...
lower_limit = ...

transform_train = transforms.Compose([
])

transform_test = transforms.Compose([
])

trainset = CIFAR10(root='./data', train=True, download=True, transform=transform_train)
trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)

testset = CIFAR10(root='./data', train=False, download=True, transform=transform_test)
testloader = DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)


classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')


Files already downloaded and verified
Files already downloaded and verified


# Defensive Distillation (25 points)

[Defensive distillation](https://arxiv.org/abs/1511.04508) proceeds in four steps:

1.   **Train the teacher network**, by setting the temperature of the softmax to T during the
training phase.
2.   **Compute soft labels** by apply the teacher network to each instance in the training set, again evaluating the softmax at temperature T.
3.  **Train the distilled network** (a network with the same shape as the teacher network) on the soft labels, using softmax at temperature T.
4.  Finally, when running the distilled network at test time to classify new inputs, use temperature 1.



## Train the teacher

In [ ]:
def train_step(model, dataloader, loss_fn, optimizer, temperature):
    # TODO: Return loss and accuracy for each epoch
    pass


def train_teacher(model, n_epochs, loader=trainloader, temp=100):
    # TODO: Log the accuracy and loss for each epoch
    pass

You can use a pre-trained resnet to speed up the training process.

In [ ]:
teacher = ...

train_teacher(teacher, 15)

Epoch [1/15] - Loss: 1.4126, Accuracy: 47.42%
Epoch [2/15] - Loss: 0.9707, Accuracy: 65.88%
Epoch [3/15] - Loss: 0.7864, Accuracy: 72.52%
Epoch [4/15] - Loss: 0.6633, Accuracy: 77.08%
Epoch [5/15] - Loss: 0.5610, Accuracy: 80.64%
Epoch [6/15] - Loss: 0.4732, Accuracy: 83.71%
Epoch [7/15] - Loss: 0.3854, Accuracy: 86.77%
Epoch [8/15] - Loss: 0.3177, Accuracy: 89.14%
Epoch [9/15] - Loss: 0.2593, Accuracy: 90.99%
Epoch [10/15] - Loss: 0.2082, Accuracy: 92.86%
Epoch [11/15] - Loss: 0.1709, Accuracy: 94.10%
Epoch [12/15] - Loss: 0.1502, Accuracy: 94.73%
Epoch [13/15] - Loss: 0.1265, Accuracy: 95.66%
Epoch [14/15] - Loss: 0.1102, Accuracy: 96.20%
Epoch [15/15] - Loss: 0.0987, Accuracy: 96.56%


## Test the teacher

In [ ]:
def test_clean(model, dataloader=testloader):
    # TODO: Return the clean accuracy of the model
    pass

Print the clean accuracy of the teacher.

In [8]:
print(f'Teacher Accuracy {test_clean(teacher):.2f}%')

Teacher Accuracy 75.99%


## Train the student

In [ ]:
def distill(model, teacher, dataloader, optimizer, T):
    # TODO: Get soft labels from teacher model
    # TODO: Get student model outputs
    # TODO: Compute the distillation loss
    # TODO: Return the accuracy (on real labels) and loss (on soft labels)
    pass


def train_student(model, teacher, n_epochs, loader=trainloader, temp=100):
    # TODO: Log the accuracy and loss for each epoch
    pass

This time use a `resnet18` without the pretrained weights.

In [ ]:
student = ...

train_student(student, teacher, 15)

Epoch [1/15] - Loss: 1.4034, Accuracy: 43.46%
Epoch [2/15] - Loss: 0.9252, Accuracy: 64.25%
Epoch [3/15] - Loss: 0.7368, Accuracy: 71.35%
Epoch [4/15] - Loss: 0.6119, Accuracy: 75.92%
Epoch [5/15] - Loss: 0.5219, Accuracy: 79.37%
Epoch [6/15] - Loss: 0.4345, Accuracy: 82.57%
Epoch [7/15] - Loss: 0.3652, Accuracy: 85.13%
Epoch [8/15] - Loss: 0.2998, Accuracy: 87.57%
Epoch [9/15] - Loss: 0.2496, Accuracy: 89.41%
Epoch [10/15] - Loss: 0.2028, Accuracy: 91.32%
Epoch [11/15] - Loss: 0.1734, Accuracy: 92.34%
Epoch [12/15] - Loss: 0.1441, Accuracy: 93.61%
Epoch [13/15] - Loss: 0.1309, Accuracy: 94.19%
Epoch [14/15] - Loss: 0.1206, Accuracy: 94.56%
Epoch [15/15] - Loss: 0.1085, Accuracy: 94.91%


## Test the student

In [12]:
print(f'Student Accuracy {test_clean(student):.2f}%')

Student Accuracy 75.36%


# Attack (15 points)

Implement the FGSM attack and the `test_attack` funcion to report the robust accuracy for different values of epsilon.

In [ ]:
def attack_fgsm(model, x, y, epsilon):
    # TODO: Return perturbed input
    pass


def attack_pgd(model, x, y, epsilon, alpha=0.2, num_iters=10):
    # TODO: Return perturbed input
    pass


def test_attack(model, epsilon, atttack=attack_fgsm, loader=testloader):
    # TODO: Return the robust accuracy for FGSM or PGD
    pass

Report the robust accuracy of the teacher for `ϵ = [1, 2, 4, 8, 16]`.

In [ ]:
epsilons = [1, 2, 4, 8, 16]

for eps in epsilons:
    # TODO:
    print(f'FGSM with ϵ={eps}/255 has Accuracy: {acc:.2f}%')
    print(f'PGD  with ϵ={eps}/255 has Accuracy: {acc:.2f}%')

FGSM with ϵ=1/255 has Accuracy: 51.15%
PGD  with ϵ=1/255 has Accuracy: 49.70%
FGSM with ϵ=2/255 has Accuracy: 32.96%
PGD  with ϵ=2/255 has Accuracy: 30.80%
FGSM with ϵ=4/255 has Accuracy: 14.21%
PGD  with ϵ=4/255 has Accuracy: 12.54%
FGSM with ϵ=8/255 has Accuracy: 3.65%
PGD  with ϵ=8/255 has Accuracy: 0.31%
FGSM with ϵ=16/255 has Accuracy: 1.27%
PGD  with ϵ=16/255 has Accuracy: 0.00%


Do the same for the student:

In [ ]:
for eps in epsilons:
    # TODO:
    print(f'FGSM with ϵ={eps}/255 has Accuracy: {acc:.2f}%')
    print(f'PGD  with ϵ={eps}/255 has Accuracy: {acc:.2f}%')

FGSM with ϵ=1/255 has Accuracy: 69.25%
PGD  with ϵ=1/255 has Accuracy: 69.17%
FGSM with ϵ=2/255 has Accuracy: 69.22%
PGD  with ϵ=2/255 has Accuracy: 69.18%
FGSM with ϵ=4/255 has Accuracy: 69.21%
PGD  with ϵ=4/255 has Accuracy: 69.20%
FGSM with ϵ=8/255 has Accuracy: 69.19%
PGD  with ϵ=8/255 has Accuracy: 69.19%
FGSM with ϵ=16/255 has Accuracy: 69.22%
PGD  with ϵ=16/255 has Accuracy: 69.19%


What do you see?

`your response:` 

# Transferring Adversarial Examples (15 points)

Train yet another model to be used as the surrogate. (set temperature to 1)

In [ ]:
model = ...

Epoch [1/15] - Loss: 2.1782, Accuracy: 25.50%
Epoch [2/15] - Loss: 1.5049, Accuracy: 45.60%
Epoch [3/15] - Loss: 1.2974, Accuracy: 53.38%
Epoch [4/15] - Loss: 1.0910, Accuracy: 61.16%
Epoch [5/15] - Loss: 0.9514, Accuracy: 66.40%
Epoch [6/15] - Loss: 0.8288, Accuracy: 70.95%
Epoch [7/15] - Loss: 0.7388, Accuracy: 74.02%
Epoch [8/15] - Loss: 0.6808, Accuracy: 76.06%
Epoch [9/15] - Loss: 0.5963, Accuracy: 79.04%
Epoch [10/15] - Loss: 0.5322, Accuracy: 81.17%
Epoch [11/15] - Loss: 0.4748, Accuracy: 83.38%
Epoch [12/15] - Loss: 0.4290, Accuracy: 84.86%
Epoch [13/15] - Loss: 0.3754, Accuracy: 86.73%
Epoch [14/15] - Loss: 0.3300, Accuracy: 88.50%
Epoch [15/15] - Loss: 0.2920, Accuracy: 89.65%


Print the surrogate accuracy.

In [18]:
print(f'Surrogate Accuracy {test_clean(surrogate):.2f}%')

Surrogate Accuracy 72.76%


Report the accuracy of the surrogate for `ϵ = [1, 2, 4, 8, 16]`.

FGSM with ϵ=1/255 has Accuracy: 45.56%
FGSM with ϵ=2/255 has Accuracy: 27.29%
FGSM with ϵ=4/255 has Accuracy: 10.49%
FGSM with ϵ=8/255 has Accuracy: 2.65%
FGSM with ϵ=16/255 has Accuracy: 1.57%


Implement the following functions to transfer attacks from a surrogate model to an oracle.

In [ ]:
def transfer_attack(oracle, model, eps, loader=testloader):
    # TODO: Attack the model and report the accuracy of the oracle
    pass

Transfer attacks for `ϵ = [1, 2, 4, 8, 16]` from your model to the student.

In [21]:
for eps in epsilons:
    acc = transfer_attack(student, surrogate, eps*scale/255)
    print(f'FGSM with ϵ={eps}/255 has Accuracy: {acc:.2f}%')

FGSM with ϵ=1/255 has Accuracy: 71.71%
FGSM with ϵ=2/255 has Accuracy: 68.10%
FGSM with ϵ=4/255 has Accuracy: 60.58%
FGSM with ϵ=8/255 has Accuracy: 44.87%
FGSM with ϵ=16/255 has Accuracy: 24.42%


- What can be inferred from these results?
- How are the accuracies of the student and the surrogate under attack related?
- Does Defensive Distillation obfuscate the gradients? Why?

`your response:`


# ZOO Based Black-Box Attacks (25 points)

Based on [Black-box Adversarial Attacks with Limited Queries and Information](https://arxiv.org/abs/1804.08598) you must first calculate the estimate of the graidents, and next attack the model based on your estimates.

In [ ]:
def nes_gradient_estimate(model, x, y, epsilon, num_samples, sigma):
    # TODO: Return the estimated gradient
    pass

I used 3 different things to estimate gradiant and all of them end up almost the same result. The bottom result is made with probabilities.

In [ ]:
def partial_information_attack(model, x, y, epsilon, num_samples, sigma, num_steps, alpha):
    # TODO: Return the perturbed image
    pass

Now run this attack on your models and report the results. (You **DON'T** need to run the attack for the entire test dataset as this will take a lot of time!)

In [25]:
epsilons = [1, 2, 4, 8, 16]

for eps in epsilons:
    acc = test_zoo_attack(model=surrogate, epsilon=eps*scale/255, num_samples=100, sigma=0.001, num_steps=10, alpha=0.1, loader=testloader)
    print(f'ZOO with ϵ={eps}/255 has Accuracy: {acc:.2f}%')

ZOO with ϵ=1/255 has Accuracy: 67.87%
ZOO with ϵ=2/255 has Accuracy: 62.30%
ZOO with ϵ=4/255 has Accuracy: 47.56%
ZOO with ϵ=8/255 has Accuracy: 15.53%
ZOO with ϵ=16/255 has Accuracy: 2.34%


# Adversarially Robust Distillation (15 points)

In this section we are going to test another type of distillation to see if this method is robust. This technique is [Adversarially Robust Distillation](https://arxiv.org/abs/1905.09747).



1.   We will try to distill a robsut teacher from [Robust Bench](https://robustbench.github.io/) onto a smaller architecture.
2.   We minimize the KL-Divergence between the logits of the student and teacher to ensure fidelity. (You can also incorporate the classification loss as mentioned in the paper but you can choose to ignore it as well)
3.   At each step of the distillation you will attack the student (you can use either FGSM or PGD) and find an adversarial example $X + \delta$ for data point $X$. Next you will minimize $t^2 \times \text{KL}(S(X+\delta), T(X))$ where $S$ and $T$ are the student and teacher networks respectively.



In [ ]:
! pip install git+https://github.com/RobustBench/robustbench.git

In [ ]:
from robustbench.utils import load_model

teacher = load_model(model_name='Gowal2021Improving_R18_ddpm_100m', dataset='cifar10', threat_model='Linf')

In [ ]:
def ard(student, teacher, dataloader, optimizer, eps, attack):
    # TODO
    pass


def adv_train_student(model, teacher, n_epochs, eps=8/255, loader=trainloader):
    # TODO
    pass

In [ ]:
student = mobilenet_v2(weights=None)

# TODO: Adjust and train the student


Epoch [1/15] - Loss: 2.8294, Accuracy: 17.19%
Epoch [2/15] - Loss: 2.3497, Accuracy: 21.36%
Epoch [3/15] - Loss: 2.1978, Accuracy: 23.50%
Epoch [4/15] - Loss: 2.1297, Accuracy: 25.51%
Epoch [5/15] - Loss: 2.0597, Accuracy: 26.75%
Epoch [6/15] - Loss: 2.0010, Accuracy: 27.47%
Epoch [7/15] - Loss: 1.9563, Accuracy: 29.34%
Epoch [8/15] - Loss: 1.9193, Accuracy: 30.78%
Epoch [9/15] - Loss: 1.9044, Accuracy: 31.92%
Epoch [10/15] - Loss: 1.8660, Accuracy: 31.20%
Epoch [11/15] - Loss: 1.8262, Accuracy: 32.59%
Epoch [12/15] - Loss: 1.8063, Accuracy: 34.40%
Epoch [13/15] - Loss: 1.7858, Accuracy: 34.44%
Epoch [14/15] - Loss: 1.7387, Accuracy: 35.12%
Epoch [15/15] - Loss: 1.7224, Accuracy: 34.12%


Now report the accuracy of the student on the test dataset.

In [ ]:
# TODO: Clean accurcy

# TODO: FGSM with eps=8/255

# TODO: PGD with eps=8/255


Student Accuracy 30.87%
FGSM with ϵ=8/255 has Accuracy: 17.19%
PGD  with ϵ=8/255 has Accuracy: 16.07%
